<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/adk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ADK Multi-Agent Pipeline with Vertex Services**

###### *Setup*

In [ ]:
############################################
# Installs and Imports
############################################

!pip install google-adk mcp litellm -q
#!pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk]>=1.112
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.126.1" "google-adk>=1.18.0"

# Miscellaneous Tools
import warnings
warnings.filterwarnings('ignore')
from google.genai import types

# Agents
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent

# ADK Tools (https://google.github.io/adk-docs/tools/)
from google.adk.tools import google_search, url_context, VertexAiSearchTool

# Session and Memory Services (https://google.github.io/adk-docs/sessions/memory/#configuration)
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.tools import load_memory

# Test Runner
import os
import asyncio
from google.adk.runners import Runner

# Vertex Services (general)
import vertexai
from vertexai import agent_engines

# Deployment, Traceing and Telemetry
from vertexai.preview import reasoning_engines

# Third party model
from google.adk.models.lite_llm import LiteLlm

###### *Environmental Variables*

In [ ]:
############################################
# Environmental Variables
############################################

# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "lunar-352813"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "lunar-352813"
os.environ["VERTEXAI_LOCATION"] = "us-central1"

# Initialize Vertex AI client
client = vertexai.Client(project="lunar-352813", location="us-central1")

In [ ]:
############################################
# Vertex AI Services
############################################

# Vertex AI Search (Grounding) - Discovery Engine API is underlying engine, includes streamAssist
DATA_STORE_ID = "projects/lunar-352813/locations/global/collections/default_collection/dataStores/gemini-news-memory_1769533223752"

# Vertex AI Session Service (Chat History)
session_service = VertexAiSessionService(
    project="lunar-352813",
    location="us-central1",
    agent_engine_id="7707203776267943936"
)

# Vertex AI Memory Bank (User Facts)
memory_service = VertexAiMemoryBankService(
    project="lunar-352813",
    location="us-central1",
    agent_engine_id="7707203776267943936")

# Volatile (RAM) Session and Memory
# session_service = InMemorySessionService()
# memory_service = InMemoryMemoryService()

In [ ]:
############################################
# Vertex AI Model as a Service (MaaS))
############################################

# Llama 4 Maverick
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/llama/llama4-maverick?hl=de
llama = LiteLlm(
    model="vertex_ai/meta/llama-4-maverick-17b-128e-instruct-maas",
    #model="vertex_ai/projects/lunar-352813/locations/us-central1/endpoints/.." # as endpoint deployed instead of fully managed API
    vertex_project="lunar-352813",
    vertex_location="us-east5"    # some models are in another region!
)

# DeepSeek V3.2 MaaS on Vertex AI
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32
deepseek = LiteLlm(
    model="vertex_ai/deepseek-ai/deepseek-v3.2-maas",
    vertex_project="lunar-352813",
    vertex_location="global"
)

# Kimi K2
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32
kimi = LiteLlm(
    model="vertex_ai/kimi/kimi-k2-thinking-maas",
    vertex_project="lunar-352813",
    vertex_location="global"
)

# Qwen 3
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/qwen/qwen3-next-thinking
qwen = LiteLlm(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas",
    vertex_project="lunar-352813",
    vertex_location="global"
)

# Gemma 3
# https://pantheon.corp.google.com/vertex-ai/publishers/google/model-garden/gemma3
gemma = LiteLlm(
    model="vertex_ai/gemma3/gemma-3-1b-it",
    vertex_project="lunar-352813",
    vertex_location="us-central1"
)

###### *Colab: Define and Test Agents  (mit OSS)*

In [ ]:
############################################
# Vertex AI Agent Engine (--> create once!)
############################################

# for VertexAiSessionService and VertexAiMemoryBankService
# ---> DO NOT START THIS AGAIN !!!!

"""
agent_engine = client.agent_engines.create(
    config={
        "display_name": "Research Agent Engine",
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    # This model handles memory extraction and consolidation
                    "model": "projects/lunar-352813/locations/us-central1/publishers/google/models/gemini-2.5-flash"
                }
            }
        }
    }
)

# Extract ID for code
print(f"Your Agent Engine ID: {agent_engine.api_resource.name.split('/')[-1]}")
"""

INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=lunar-352813&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%227707203776267943936%22.
INFO:vertexai_genai.agentengines:Agent Engine created. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/892203813305/locations/us-central1/reasoningEngines/7707203776267943936')


Your Agent Engine ID: 7707203776267943936


In [ ]:
############################################
# Agents and Workflow
############################################

# 1. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-2.5-flash',
    description=('Use GoogleSearchTool to find news latest news about Gemini and ChatGPT. Return only a list of URLs.'),
    instruction='Search for the latest public news on the topic.',
    tools=[google_search],
    output_key='search_results'
)

# 2. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-2.5-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)], # Vertex AI Search with DATA_STORE_ID
    output_key='internal_data'
)

# 3. Parallel Retrieval Agent
parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, doc_agent]
)

# 4. Web Search Analysis Agent (Processes raw URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-2.5-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    output_key='web_analysis'
)

# 5. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    #model='gemini-2.5-pro',
    model=deepseek,
    #model=llama,
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    output_key='final_comparison'
)

# 6. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-2.5-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
)

# 7. Root Orchestrator (Master Flow)
root_agent = SequentialAgent(
    name='researcher',
    # no model needed
    # no instructions needed, it just moves data from 0 -> 1 -> 2 -> 3
    sub_agents=[parallel_retrieval, search_analyst, analyst, summarizer],
)

In [ ]:
############################################
# Test Agent (Colab Runtime)
############################################

APP_NAME="hybrid_news_app"
USER_ID="deltorobarba"

# Create persistent session + get session ID from Vertex AI:
new_session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
)
active_session_id = new_session.id
print(f"New persistent session started with ID: {active_session_id}")

### Create hardcoded session ID only when volatile 'session_service = InMemorySessionService()' used
# active_session_id="session_01"
### Register custom ID before Runner can use it:
#await session_service.create_session(
#    app_name=APP_NAME,
#    user_id=USER_ID,
#    session_id=active_session_id)
#print(f"Volatile session started with custom ID: {active_session_id}")

# Formulate query for agents
query = "I'm CTO at Uluru Bank in Australia. Search for the latest 2026 news on Gemini 3 and compare its capabilities against GPT-4o."

# Initialize Runner
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service # Connect to agent engine with agent_engine_id
    )

async def run_hybrid_flow():
    user_msg = types.Content(
        role="user",
        parts=[types.Part(text=query)])

    print(f"✅ Starting {root_agent.name} agent ..\n")

    # Modellnamen
    if hasattr(analyst.model, 'model'):
        # Case: LiteLlm Objekt
        model_name = analyst.model.model.split('/')[-1]
    else:
        # Case: Direkter String (z.B. bei Gemini)
        model_name = analyst.model
    print(f"Model for analyst agent: {model_name}\n")

    # Run agents
    async for event in runner.run_async(user_id=USER_ID, session_id=active_session_id, new_message=user_msg):
        if event.is_final_response() and event.content and event.author == "summarizer":
            print(f"{event.content.parts[0].text}")

    # Save to Memory
    updated_session = await runner.session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=active_session_id)

    # Sends entire chat transcript to specialized "Extractor" model. Reads conversation, extracts unique facts and save them.
    await memory_service.add_session_to_memory(updated_session)
    print("\n✅ [System] Finished. Memory extraction triggered.")

# Run the consolidated function
await run_hybrid_flow()

New persistent session started with ID: 2785771176083849216
✅ Starting researcher agent ..

Model for analyst agent: deepseek-v3.2-maas

Here's a summary for the CEO:

*   **Conflicting Information:** There's a major contradiction regarding Gemini 3; our internal specialist finds no public news, while web reports detail its release, Google integration, and advanced features like 1M+ token context windows and agentic coding.
*   **GPT-4o Retirement Imminent:** GPT-4o is reportedly retiring on February 13, 2026, meaning Uluru Bank needs an immediate migration plan if we currently use it.
*   **Hypothetical Gemini 3 Advantages:** If the web reports are accurate, Gemini 3 offers superior context windows, advanced video processing, and agentic capabilities highly beneficial for financial services.
*   **Critical Availability Gap:** The primary concern is the unconfirmed public availability of Gemini 3. Relying on unverified web information for strategic decisions carries high risk.
*   **Ur

###### *Agent Engine: Define, Deploy and Test Agents*

In [ ]:
############################################
# Define Agents and Workflow
############################################

# 1. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-2.5-flash',
    description=('Use GoogleSearchTool to find news latest news about Gemini and ChatGPT. Return only a list of URLs.'),
    instruction='Search for the latest public news on the topic.',
    tools=[google_search],
    output_key='search_results'
)

# 2. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-2.5-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)], # Vertex AI Search with DATA_STORE_ID
    output_key='internal_data'
)

# 3. Parallel Retrieval Agent
parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, doc_agent]
)

# 4. Web Search Analysis Agent (Processes raw URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-2.5-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    output_key='web_analysis'
)

# 5. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    model='gemini-2.5-pro',
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    output_key='final_comparison'
)

# 6. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-2.5-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
)

# 7. Root Orchestrator (Master Flow)
root_agent = SequentialAgent(
    name='researcher',
    # no model needed
    # no instructions needed, it just moves data from 0 -> 1 -> 2 -> 3
    sub_agents=[parallel_retrieval, search_analyst, analyst, summarizer],
)

In [ ]:
############################################
# Deploy Agents Google Cloud (modern, with agentengine framework)
############################################

"""
Permissions: Take the AI Platform Reasoning Engine Service Agent that the agent uses:
'service-892203813305@gcp-sa-aiplatform-re.iam.gserviceaccount.com'.
Go to IAM, enable 'Include Google-provided role grants' and add role: 'Discovery Engine Viewer', 'Vertex AI User' and 'Cloud Trace Agent' (write)
Troubleshooting: https://docs.cloud.google.com/agent-builder/agent-engine/troubleshooting/use

Traceing and Logging:
* Cloud Logging API in GCP project aktivieren
* Telemetry API in GCP project aktivieren
* Add role 'Cloud Trace Agent' (write) to service account
* https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing
* https://docs.cloud.google.com/agent-builder/agent-engine/deploy
"""

# Create GCS bucket first if you haven't already
STAGING_BUCKET = "gs://lunar-352813-agent-staging"

# https://google.github.io/adk-docs/deploy/agent-engine/deploy/#setup-cloud-project
# https://docs.cloud.google.com/agent-builder/agent-engine/develop/custom
# Environmental Variables for Traceing (https://docs.cloud.google.com/agent-builder/agent-engine/deploy)
# Set env variables (https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing)
custom_env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

# 1. Initialize the client
client = vertexai.Client(project="lunar-352813", location="us-central1")

# 2. Wrap the agent
adk_app = reasoning_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
    env_vars=custom_env_vars
)

# 3. Deploy
remote_agent = client.agent_engines.create(
    agent=adk_app,
    config={
        # This tells Vertex AI where to upload your agent's code
        "staging_bucket": STAGING_BUCKET,

        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]>=1.135",
            "google-adk>=1.23.0",
            "google-cloud-trace"
        ],
        "env_vars":custom_env_vars,
        "display_name":"Research Agent Engine (Live Llama)",
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    "model": f"projects/lunar-352813/locations/us-central1/publishers/google/models/gemini-2.5-flash"
                }
            }
        }
    },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'cloudpickle': '3.1.0', 'pydantic': '2.12.5', 'google-cloud-aiplatform': '1.135.0'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.0', 'pydantic==2.12.5'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.135', 'google-adk>=1.23.0', 'google-cloud-trace', 'cloudpickle==3.1.0', 'pydantic==2.12.5']
INFO:vertexai_genai.agentengines:Using bucket lunar-352813-agent-staging
INFO:vertexai_genai.agentengines:Wrote to gs://lunar-352813-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://lunar-352813-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://lunar-352813-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framew

In [ ]:
############################################
# Test Agent (Single Query)
############################################

# 1. Initialize the client
client = vertexai.Client(project="lunar-352813", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/892203813305/locations/us-central1/reasoningEngines/2753672995695230976"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_remote_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Create session and grab the 'id' key specifically 🔑
    raw_session = await remote_app.async_create_session(user_id=user_id)

    # Based on your last error, the key is definitely 'id'
    session_id = raw_session.get('id')

    if not session_id:
        print(f"❌ Error: ID not found in response: {raw_session}")
        return

    print(f"✅ Created Managed Session ID: {session_id}")

    # 4. Stream the query (nur Summarizer-Events anzeigen) 🌊
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message="Uluru bank uses ChatGPT. The annual revenue of Uluru bank is 500 Mio USD. Search news on Gemini 3 and compare it to GPT-4o."
    ):
        # Filtern nach dem Author 'summarizer' 🔍
        # if 'content' in event: # print every agent's output
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Trigger Memory Extraction 🧠
    print(f"\n[System] Fetching session state for extraction...")
    current_session = await remote_app.async_get_session(
        user_id=user_id,
        session_id=session_id
    )

    # Check for events to ensure history is captured
    # Docs: https://google.github.io/adk-docs/events/#extracting-key-information
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} events. Sending to Memory Bank...")
        # Pass the dictionary directly to the memory service
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered! Check 'Gemerkte Informationen' in 1-2 mins.")
    else:
        print("\n❌ Error: Session history was empty. No memories extracted.")

# Execute
await run_remote_test()

Connecting to: projects/892203813305/locations/us-central1/reasoningEngines/2753672995695230976...
✅ Created Managed Session ID: 1822255942524207104

--- Agent Output (Summary Agent) ---
Here are 5 punchy bullets comparing Gemini 3 and GPT-4o for the CEO:

*   **Contradiction Alert: Quantum Telepathy Vision.** Internal briefings hint at a future/conceptual Gemini 3 with "quantum telepathy" for quantum data processing—a truly unique, market-disrupting capability. However, this is *not* mentioned in public announcements for Gemini 3's upcoming late 2025 release.
*   **Massive Context Window & Fresh Data.** Gemini 3 Flash offers a 1 Million token context (8x GPT-4o's 128K) and vastly fresher training data (January 2025 vs. October 2023), allowing superior information processing.
*   **Autonomous Software Development & Enhanced Security.** Gemini 3 includes "Google Antigravity" for autonomous software development and significantly improved security against prompt injections and sycophancy.

In [ ]:
############################################
# Test Agent (Follow Up Query)
############################################

import vertexai

# 1. Initialize the client
client = vertexai.Client(project="lunar-352813", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/892203813305/locations/us-central1/reasoningEngines/2753672995695230976"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_follow_up():
    print(f"Resuming Session on: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Explicitly reuse your Session ID from the first run! <-------- copy/paste existing session ID
    session_id = "1822255942524207104"

    print(f"✅ Reusing Managed Session ID: {session_id}")

    # 4. Stream the follow-up query 🌊
    # The agent will remember the Uluru Bank details from the first query.
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message="How can I use Gemini 3 in financial services"
        # message="Based on that revenue, which Gemini 3 model (Pro or Flash) is more cost-effective for us?"
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Optional: Trigger Memory Extraction again 🧠
    # This will now include the context from both queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"\n✅ Found {len(current_session['events'])} total events across both turns.")
        await remote_app.async_add_session_to_memory(session=current_session)

# Execute the second run
await run_follow_up()

Resuming Session on: projects/892203813305/locations/us-central1/reasoningEngines/2753672995695230976...
✅ Reusing Managed Session ID: 1822255942524207104

--- Agent Output (Summary Agent) ---
Uluru Bank can leverage Gemini 3's advanced capabilities across its operations to significantly enhance efficiency, strategy, and security:

*   **Supercharge Compliance & Risk Management:** Utilize Gemini 3's 1 million-token context window to analyze thousands of pages of contracts, regulatory filings, and transactional data in minutes. This will drastically improve fraud detection, identify subtle risks, and ensure compliance with unprecedented speed and accuracy, aided by its enhanced security features.
*   **Revolutionize Customer Experience & Advisory:** Deploy Gemini 3's multimodal capabilities (text, audio, video) for highly personalized financial advisory and customer service. It can analyze complete client histories to offer tailored advice and power sophisticated, natural-sounding chatb

In [ ]:
############################################
# Test Agent (Multi-Turn Query)
############################################

async def run_multi_turn_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # --- Turn 1: Initial Query (Creates the Session) ---
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get('id')
    print(f"✅ Session Created: {session_id}")

    # First Message
    await execute_query(user_id, session_id, "Uluru bank uses ChatGPT. Annual revenue is 500 Mio USD. Compare Gemini 3 to GPT-4o.")

    # --- Turn 2: Follow-up Query (Reuses the Session ID) ---
    print(f"\n--- Starting Follow-up Query in Session: {session_id} ---")

    # Notice we use the SAME session_id variable here!
    await execute_query(user_id, session_id, "Based on that revenue, which model is more cost-effective for us?")

    # --- Step 5: Final Memory Extraction ---
    # Fetching the session now will show events from BOTH queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} total events. Extracting to Memory Bank...")
        await remote_app.async_add_session_to_memory(session=current_session)

# Helper function to keep the code clean
async def execute_query(user_id, session_id, message):
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=message
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n[Summarizer]: {part['text']}")

await run_multi_turn_test()

Connecting to: projects/892203813305/locations/us-central1/reasoningEngines/2753672995695230976...
✅ Session Created: 6978877515863425024

[Summarizer]: Here are 5 key points for the CEO:

*   **Clarifying Gemini:** While "Gemini 3" is often a conceptual term (e.g., "quantum telepathy"), Google's current *Gemini 3 Pro* and *2.5 Pro* are available and offer significant, immediate upgrades over GPT-4o for Uluru Bank's operations.
*   **Unmatched Data Insight:** Gemini's superior reasoning and vast 1M+ token context window (vs. GPT-4o's 128K) enable deeper, contextual analysis of entire financial reports, legal documents, and market trends, providing critical insights.
*   **Accelerated Innovation:** Gemini excels in code generation, capable of building entire internal applications and interactive UIs from a single prompt, drastically speeding up development of bespoke banking tools and dashboards.
*   **Precision & Multimodality:** Features like "Deep Think" mode and "best-in-world" mult

###### *MCP and BigQuery*

ADK Code Assistant
Custom Gem
This is going to be fun! 🛠️ We are building a real microservice architecture inside a notebook.

Here is the plan for this step:

1. Create the Server Code: We will write a file bq_server.py that defines our BigQuery tool and wraps it in a web server (Starlette + Uvicorn).

2. Launch the Server: We will use a background process to start running it on port 8000.

In [ ]:
# Bigquery authentification
from google.colab import auth
# 1. Authenticate (Pop-up will appear)
auth.authenticate_user()

# 2. Install the web server and BigQuery tools
!pip install uvicorn sse-starlette google-cloud-bigquery

In [ ]:
%%writefile bq_server.py
import uvicorn
from starlette.applications import Starlette
from starlette.responses import Response
from starlette.routing import Route
from mcp.server.sse import SseServerTransport
from mcp.server import Server
import mcp.types as types
from google.cloud import bigquery

# 1. Initialize Server
server = Server("bigquery-analyst")

# 2. Define Tool Logic
def execute_query(sql_query: str) -> str:
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    if any(cmd in sql_query.upper() for cmd in forbidden):
        return "Error: Modification queries are not allowed. Read-only access."

    try:
        client = bigquery.Client()
        query_job = client.query(sql_query)
        # Limit to 20 rows to avoid blowing up the context window
        results = [dict(row) for row in query_job.result()]

        if not results:
            return "Query executed successfully but returned no results."

        return str(results[:20])
    except Exception as e:
        return f"BigQuery Error: {str(e)}"

# 3. Register Tool
@server.list_tools()
async def handle_list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="query_public_dataset",
            description="Run a read-only SQL query against BigQuery public datasets.",
            inputSchema={
                "type": "object",
                "properties": {
                    "sql_query": {"type": "string", "description": "The SQL query to execute"}
                },
                "required": ["sql_query"]
            }
        )
    ]

@server.call_tool()
async def handle_call_tool(name: str, arguments: dict | None) -> list[types.TextContent | types.ImageContent | types.EmbeddedResource]:
    if name == "query_public_dataset":
        if not arguments or "sql_query" not in arguments:
            raise ValueError("Missing sql_query argument")
        result = execute_query(arguments["sql_query"])
        return [types.TextContent(type="text", text=result)]
    raise ValueError(f"Unknown tool: {name}")

# 4. Web Server Logic
sse = SseServerTransport("/messages")

async def handle_sse(request):
    async with sse.connect_sse(request.scope, request.receive, request._send) as streams:
        await server.run(streams[0], streams[1], server.create_initialization_options())
    return Response()

async def handle_messages(request):
    # FIX: Unpack the request into raw ASGI components
    await sse.handle_post_message(request.scope, request.receive, request._send)

# Define routes
routes = [
    Route("/sse", endpoint=handle_sse),
    Route("/messages", endpoint=handle_messages, methods=["POST"])
]

app = Starlette(debug=True, routes=routes)

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting bq_server.py


In [ ]:
import subprocess
import time
import requests

!fuser -k 8000/tcp > /dev/null 2>&1

print("🚀 Restarting BigQuery MCP Server (Fixed POST)...")
with open("server.log", "w") as log:
    proc = subprocess.Popen(["python", "bq_server.py"], stdout=log, stderr=log)

print("Waiting for initialization...")
time.sleep(5)

try:
    response = requests.get("http://localhost:8000/sse", stream=True, timeout=2)
    print(f"✅ Server is UP! (Status: {response.status_code})")
except requests.exceptions.Timeout:
    print("✅ Server is UP and listening!")
except Exception as e:
    print(f"❌ Server check failed: {e}")

🚀 Restarting BigQuery MCP Server (Fixed POST)...
Waiting for initialization...
✅ Server is UP! (Status: 200)


In [ ]:
import time
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset # Ensure this is imported

# 1. Define Connection
bq_params = SseConnectionParams(
    url="http://0.0.0.0:8000/sse"
)

# 2. Create Toolset & Agent
mcp_bq_tools = MCPToolset(connection_params=bq_params)

enterprise_agent = LlmAgent(
    name='bigquery_analyst',
    model='gemini-2.0-flash',
    instruction="""
    You are an Enterprise Data Analyst.
    You have access to Google BigQuery public datasets.
    When asked a question:
    1. Formulate a correct BigQuery SQL query.
    2. Use the 'query_public_dataset' tool to execute it.
    3. Summarize the findings for the user.
    """,
    tools=[mcp_bq_tools]
)

# 3. Setup Runner
APP_NAME = "bq_demo_app"
active_session_id = f"bq_session_{int(time.time())}"

bq_runner = Runner(
    agent=enterprise_agent,
    session_service=session_service,
    app_name=APP_NAME,
)

# 4. Create Session
try:
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=active_session_id
    )
    print(f"✅ Created new session: {active_session_id}")
except Exception:
    print(f"🔄 Using existing session: {active_session_id}")

# 5. Ask the Question
question = "What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table."

print(f"🤖 User: {question}\n")

user_msg = types.Content(
    role="user",
    parts=[types.Part(text=question)]
)

async for event in bq_runner.run_async(
    user_id=USER_ID,
    session_id=active_session_id,
    new_message=user_msg
):
    if event.is_final_response() and event.content:
        print(f"🕵️ Analyst: {event.content.parts[0].text}")

✅ Created new session: bq_session_1769812172
🤖 User: What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table.

🕵️ Analyst: The top 3 most popular female names in the USA in 1980 were Jennifer, Amanda, and Jessica.


In [ ]:
# Check the logs to see why the server crashed on startup
try:
    with open("server.log", "r") as f:
        print("--- SERVER LOGS START ---")
        print(f.read())
        print("--- SERVER LOGS END ---")
except FileNotFoundError:
    print("❌ Could not find server.log")